# NB-02 · Ingest & Chunk

**Tujuan:** Mengubah dokumen nyata (PDF) menjadi potongan teks (*chunk*) yang siap di-*retrieve* oleh sistem RAG.

## Kenapa *chunking* penting?

- **Konteks LLM terbatas.** Model bahasa memiliki batas panjang konteks (misalnya 4 096–128 000 token). Kita tidak bisa memasukkan seluruh dokumen ke dalam satu *prompt*.
- **Retrieval bekerja per-potongan.** Saat pengguna mengajukan pertanyaan, sistem mencari potongan yang paling relevan — bukan seluruh dokumen. Potongan yang terlalu panjang menurunkan presisi; potongan yang terlalu pendek kehilangan konteks.
- **Strategi berbeda, hasil berbeda.** Tidak ada satu strategi *chunking* yang selalu terbaik. Notebook ini membandingkan tiga pendekatan dan mengukur kualitasnya secara kuantitatif.

In [ ]:
# Install ingestion dependencies
!pip install -q docling pdfplumber sentence-transformers "transformers<5"

In [ ]:
# --- Navasena course bootstrap (Colab) ---
import os, sys
REPO = "navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chmdznr/navasena-gen-ml-course.git
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))
from tools.rag_utils import TokenCounter, TextChunker, chunk_quality_score, split_sentences

## Kenapa ingest PDF itu sulit? (bukan sekadar 'PDF → teks')

PDF menyimpan **bagaimana halaman terlihat** (glyph pada koordinat x,y), bukan bagaimana ia **dibaca**. Akibatnya extractor naif tidak punya konsep kolom, tabel, atau header. Lima jebakan klasik:

1. **Kolom ganda → 'word salad'.** Pembacaan kiri-ke-kanan menempel baris kolom kiri dengan kolom kanan jadi kalimat ngawur.
2. **Tabel → angka lepas dari header.** `Pendapatan 1200 1500 1700` kehilangan kaitan angka dengan kolomnya, fakta jadi tak terjawab.
3. **PDF hasil scan → string kosong, senyap.** Tidak ada text layer, jadi halaman itu menyumbang nol chunk tanpa error apa pun.
4. **Header/footer/nomor halaman** berulang tiap halaman dan nyempil di tengah kalimat, membanjiri index dengan boilerplate.
5. **Caption lepas dari gambar; ligatur (fi/fl) & tanda hubung** memecah kata (`regu-larization` → dua potongan), sehingga kata itu hilang dari pencarian.

**Intinya (GIGO):** ingest adalah fondasi RAG. Chunking, reranking, atau prompt sehebat apa pun **tidak bisa** memperbaiki teks yang sudah rusak saat ekstraksi.

## Docling vs Ekstraksi Naif

Docling menyelesaikan kelima jebakan di atas dengan pipeline yang tahu *struktur* dokumen, bukan sekadar urutan glyph.

| Kasus | Ekstraksi naif (PyPDF2/pdfplumber) | Docling |
|---|---|---|
| **Reading-order multi-kolom** | Baris kolom kiri + kanan bergabung jadi kalimat ngawur | Layout model mendeteksi kolom, reading-order dipulihkan |
| **Tabel** | Angka terlepas dari header kolom (`1200 1500 1700`) | TableFormer merekonstruksi grid baris×kolom → Markdown/DataFrame |
| **PDF scan** | String kosong (tidak ada text layer) | OCR membaca gambar per-region, menghasilkan teks |
| **Header/footer (furniture)** | Teks boilerplate nyempil di tengah kalimat | Label `page-header`/`page-footer` dibuang sebelum export |
| **Caption/gambar** | Caption dan gambar tercampur teks biasa | Item bertipe `picture` + `caption` dikelompokkan terpisah |
| **Output** | String datar (satu blok teks panjang) | `DoclingDocument` — pohon bertipe (texts, tables, pictures) dengan provenance |

> **Catatan jujur:** saat pertama kali `ingest()` dijalankan, Docling mengunduh model layout + TableFormer (~500 MB). Di Colab T4 ini butuh ~3–5 menit tanpa progress bar — biarkan berjalan sampai selesai, jangan dikira hang. Selain itu, karena kita inisialisasi konverter SATU kali dan menggunakannya ulang, pemanggilan `ingest()` berikutnya jauh lebih cepat (model tidak dimuat ulang).

In [ ]:
# Docling pipeline made explicit: OCR + table structure are configurable, not hidden.
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption

def build_converter():
    opts = PdfPipelineOptions()
    opts.do_ocr = True                  # OCR scanned/bitmap regions (needed for scanned PDFs)
    opts.do_table_structure = True      # rebuild row/column grid via TableFormer
    opts.table_structure_options.mode = TableFormerMode.FAST  # FAST = T4-friendly; ACCURATE is Docling's default (slower, higher fidelity)
    # For Indonesian corpora you can set OCR languages, e.g. opts.ocr_options.lang = ["id", "en"]
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})

_CONVERTER = None

def ingest(path):
    """Return (markdown, DoclingDocument | None). Reuses ONE converter so model weights load once."""
    global _CONVERTER
    try:
        if _CONVERTER is None:
            _CONVERTER = build_converter()   # built once; the first .convert() downloads the models
        doc = _CONVERTER.convert(path).document
        return doc.export_to_markdown(), doc
    except Exception as e:
        print(f"Docling gagal ({type(e).__name__}: {e}); pakai pdfplumber.")
        import pdfplumber
        with pdfplumber.open(path) as pdf:
            return "\n\n".join((pg.extract_text() or "") for pg in pdf.pages), None

# upload-or-fallback (baked-in sample so the cell always runs)
try:
    from google.colab import files
    up = files.upload()
    pdf_path = next(iter(up)) if up else None
except Exception:
    pdf_path = None
if not pdf_path:
    pdf_path = "navasena-gen-ml-course/05_rag/data/sample_id_document.pdf"

raw_text, doc = ingest(pdf_path)   # convert ONCE; reuse `doc` below
print(raw_text[:400])

In [ ]:
import pdfplumber
with pdfplumber.open(pdf_path) as pdf:
    raw_dump = "\n\n".join((pg.extract_text() or "") for pg in pdf.pages)
print("=== Docling (markdown, sadar-struktur) ===")
print(raw_text[:400])
print("\n=== pdfplumber (raw dump, datar) ===")
print(raw_dump[:400])
# Pada dokumen teks bersih 1 kolom ini keduanya mirip; bedanya MELEDAK pada tabel/kolom/scan (lihat demo di bawah).

In [ ]:
# The real payoff: a typed tree with labels + provenance (not one flat string).
if doc is not None:
    print("Elemen — texts:", len(doc.texts), "| tables:", len(doc.tables), "| pictures:", len(doc.pictures))
    print("\n10 item pertama (level | label | cuplikan):")
    for i, (item, level) in enumerate(doc.iterate_items()):
        if i >= 10:
            break
        snippet = getattr(item, "text", "")
        print(f"  {level} | {item.label} | {snippet[:50]}")
else:
    print("Docling tidak menghasilkan dokumen terstruktur (memakai fallback pdfplumber).")

## Demo Kasus Sulit: Tabel & Dokumen Scan

Pada dokumen Borobudur yang bersih dan satu kolom, perbedaan Docling vs pdfplumber nyaris tak terlihat — keduanya menghasilkan teks yang serupa. Untuk membuktikan nilai sesungguhnya, kita gunakan dua sampel khusus di `05_rag/data/`: **`sample_table.pdf`** (tabel keuangan kecil dengan beberapa kolom angka) dan **`sample_scanned.pdf`** (dokumen hasil "scan" Bahasa Indonesia, tanpa text layer — piksel murni, tidak ada karakter tersimpan di PDF).

Untuk kasus tabel, **kesuksesan** terlihat dari perbedaan output yang jelas: pdfplumber meratakan semua kolom menjadi satu baris angka yang lepas dari headernya, sementara Docling/TableFormer mengembalikan grid baris×kolom yang utuh dan bisa langsung dikonversi ke DataFrame. Untuk kasus scan, **kesuksesan** berarti: pdfplumber mengembalikan 0 karakter (dokumen benar-benar *tak terlihat* oleh index naif), sedangkan Docling+OCR menghasilkan teks Bahasa Indonesia yang terbaca.

> Sel-sel di bawah bersifat *gated*: jika file sampel belum ada, sel mencetak penjelasan konsep dan dilanjutkan tanpa crash.

In [ ]:
import os
table_pdf = "navasena-gen-ml-course/05_rag/data/sample_table.pdf"
if os.path.exists(table_pdf):
    try:
        _, tdoc = ingest(table_pdf)
        print("=== Docling: tabel direkonstruksi (TableFormer) ===")
        if tdoc is not None and tdoc.tables:
            tbl = tdoc.tables[0]
            try:
                print(tbl.export_to_dataframe().to_string(index=False))
            except Exception:
                print(tbl.export_to_markdown())
        else:
            print("(tidak ada tabel terdeteksi / memakai fallback)")
    except Exception as e:
        print(f"Demo Docling tabel dilewati ({type(e).__name__}: {e})")
    import pdfplumber
    with pdfplumber.open(table_pdf) as pdf:
        print("\n=== pdfplumber: tabel jadi teks rata / flatten ===")
        print((pdf.pages[0].extract_text() or "")[:300])
else:
    print("Sampel tabel belum ada; baca penjelasan di atas sebagai konsep.")

In [ ]:
scan_pdf = "navasena-gen-ml-course/05_rag/data/sample_scanned.pdf"
if os.path.exists(scan_pdf):
    import pdfplumber
    with pdfplumber.open(scan_pdf) as pdf:
        naive = "".join((pg.extract_text() or "") for pg in pdf.pages)
    print("pdfplumber tanpa OCR -> panjang teks:", len(naive.strip()), "(kosong: dokumen tak terlihat oleh index)")

    print("\n=== Docling dengan OCR (Bahasa Indonesia) ===")
    try:
        from docling.datamodel.pipeline_options import EasyOcrOptions
        # Set OCR ke Bahasa Indonesia agar teks scan terbaca akurat.
        # Catatan: pertama kali dijalankan, EasyOCR mengunduh model 'id' (ratusan MB) — wajar kalau lambat.
        ocr_opts = PdfPipelineOptions()
        ocr_opts.do_ocr = True
        ocr_opts.do_table_structure = False
        ocr_opts.ocr_options = EasyOcrOptions(lang=["id", "en"])
        id_conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=ocr_opts)})
        ocr_text = id_conv.convert(scan_pdf).document.export_to_markdown()
        print(ocr_text[:300] if ocr_text.strip() else "(OCR belum menghasilkan teks)")
    except Exception as e:
        print(f"Demo OCR dilewati ({type(e).__name__}: {e})")
else:
    print("Sampel scan belum ada; baca penjelasan di atas sebagai konsep.")

## Menghitung Token dengan Tokenizer Qwen

Ukuran *chunk* yang bermakna harus diukur dalam **token**, bukan karakter — karena LLM bekerja dengan token. Kita menggunakan tokenizer Qwen2.5-3B-Instruct agar ukuran *chunk* di sini konsisten dengan model *generator* yang dipakai di `nb01_rag_fundamentals.ipynb`.

`TokenCounter` adalah *wrapper* yang bisa menerima fungsi tokenisasi apa pun (*injectable*), sehingga mudah diganti jika kita berganti model.

In [ ]:
# Wire the real Qwen tokenizer into the injectable TokenCounter
from transformers import AutoTokenizer

qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
counter = TokenCounter(tokenize_fn=lambda t: len(qwen_tok.encode(t)))

# Quick sanity check
sample = raw_text[:200]
print(f"200 karakter pertama = {counter.count(sample)} token (Qwen tokenizer)")

## Tiga Strategi Chunking

**Overlap** adalah teknik di mana beberapa kata di akhir satu chunk diulang di awal chunk berikutnya — tujuannya agar konteks di batas potongan tidak hilang saat retrieval. Di kode ini parameternya adalah `overlap_words` (diukur dalam kata, bukan subword token).

### 1. `sentence` — Per-kalimat (greedy pack)
Menggabungkan kalimat-kalimat secara serakah sampai batas token tercapai. **Kelebihan:** batas *chunk* selalu di akhir kalimat (koheren). **Kapan kurang cocok:** kalimat yang sangat panjang (mis. kalimat legal atau teknis) bisa melebihi budget `max_tokens` sendirian, sehingga satu chunk otomatis over-budget.

### 2. `fixed` — Jendela kata tetap dengan overlap
Memotong teks menjadi jendela kata berukuran tetap (diukur dalam token) dengan tumpang-tindih (*overlap*) di antara *chunk* berurutan. **Kelebihan:** ukuran *chunk* sangat konsisten. **Kapan kurang cocok:** bisa memotong di tengah kalimat atau bahkan di tengah ide, sehingga satu kalimat utuh terpotong dan maknanya hilang di batas chunk.

### 3. `semantic` — Per-paragraf
Memisahkan teks berdasarkan baris kosong (pemisah paragraf alami). Paragraf yang melebihi budget di-*sub-chunk* dengan strategi `sentence`. **Kelebihan:** mempertahankan unit semantik dokumen. **Kapan kurang cocok:** sangat bergantung pada kualitas format dokumen asli — teks tanpa paragraf jelas (mis. hasil OCR baris-per-baris) menghasilkan satu chunk raksasa atau banyak micro-chunk. Overlap tidak dipakai pada strategi semantic karena antar-paragraf biasanya tidak berbagi konteks yang perlu diulang.

> **Parameter kunci:** `overlap_words` (bukan `overlap_tokens`) — diukur dalam *word*, bukan subword token.

In [ ]:
# Build chunks with all three strategies and score each one
# Note: overlap_words is measured in words, not subword tokens
sent_chunker  = TextChunker(counter, max_tokens=128, overlap_words=0)
fixed_chunker = TextChunker(counter, max_tokens=128, overlap_words=24)

results = {
    "sentence": sent_chunker.sentence(raw_text),
    "fixed":    fixed_chunker.fixed(raw_text),
    "semantic": sent_chunker.semantic(raw_text),
}

print(f"{'Strategi':10s}  {'# chunk':>8s}  {'quality score':>14s}")
print("-" * 38)
for name, chunks in results.items():
    score = chunk_quality_score(chunks, max_tokens=128, original_len=len(raw_text))
    print(f"{name:10s}  {len(chunks):>8d}  {score:>14.1f}")

## Membaca Skor Kualitas

`chunk_quality_score` menghasilkan skor 0–100 yang menggabungkan tiga aspek:

1. **Budget adherence** — seberapa banyak *chunk* yang berada dalam batas `max_tokens`. Nilai 100 berarti semua *chunk* di bawah budget.
2. **Konsistensi ukuran** — *chunk* yang seragam memudahkan sistem retrieval memperkirakan biaya konteks.
3. **Coverage** — total token semua *chunk* harus mendekati panjang dokumen asli (tidak ada teks yang hilang).

Strategi `semantic` biasanya menang untuk dokumen terstruktur (laporan, artikel) karena paragraf adalah unit makna alami. Strategi `fixed` biasanya lebih baik untuk teks yang tidak terstruktur atau dokumen yang dikonversi dari OCR dengan baris pendek-pendek.

In [ ]:
# Comparison table + token-size histogram per strategy (CPU only — no GPU needed)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, chunks) in zip(axes, results.items()):
    ax.hist([c.n_tokens for c in chunks], bins=10)
    ax.axvline(128, color="red", linestyle="--", label="max_tokens")
    ax.set_title(f"{name} (n={len(chunks)})")
    ax.set_xlabel("token per chunk")
    ax.legend()
axes[0].set_ylabel("jumlah chunk")
plt.tight_layout()
plt.show()

In [ ]:
# Docling's own chunker respects document structure (sections/tables), unlike our 3 naive string splitters.
print("=== HybridChunker: chunking sadar-struktur (pembanding ke-4) ===")
if doc is not None:
    try:
        from docling.chunking import HybridChunker
        # The chunker's tokenizer MUST match the embedding model used for retrieval.
        hchunker = HybridChunker(tokenizer="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", max_tokens=128)
        dchunks = list(hchunker.chunk(dl_doc=doc))
        print(f"{len(dchunks)} chunk sadar-struktur")
        print("\nchunk[0].text (mentah):\n", dchunks[0].text[:200])
        # contextualize() prepends the heading breadcrumb -> EMBED THIS, not .text alone:
        print("\ncontextualize(chunk[0]) (dengan breadcrumb heading):\n", hchunker.contextualize(dchunks[0])[:250])
    except Exception as e:
        print(f"HybridChunker tidak tersedia di versi Docling ini ({type(e).__name__}: {e}); lihat konsep di atas.")
else:
    print("Tidak ada DoclingDocument (fallback pdfplumber) -> HybridChunker butuh dokumen terstruktur.")

## Memilih Chunk Terbaik dan Meng-*embed*

Setelah membandingkan tiga strategi string-splitting *dan* HybridChunker sadar-struktur dari Docling, kita memilih hasil terbaik untuk dokumen ini dan mengubah setiap *chunk* menjadi vektor *embedding*. Vektor ini yang nantinya disimpan di *vector store* dan digunakan oleh `nb03` untuk retrieval.

Kita menggunakan model **`paraphrase-multilingual-MiniLM-L12-v2`** — model yang sama dengan `nb01` — agar vektor yang dihasilkan kompatibel di seluruh pipeline RAG.

> **Catatan:** Jika menggunakan HybridChunker secara penuh, embed `hchunker.contextualize(chunk)` (bukan `chunk.text`) agar heading breadcrumb ikut terbawa. `contextualize()` menambahkan breadcrumb heading ke depan chunk sehingga dua paragraf yang mirip tapi berada di bab berbeda menghasilkan embedding yang berbeda — tanpa itu keduanya nyaris identik dan retrieval bisa salah ambil.

In [ ]:
# Embed the chosen strategy's chunks with the multilingual embedder
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Pilih strategi dengan skor tertinggi (biarkan kode yang memutuskan, bukan hardcode)
best_name = max(results, key=lambda n: chunk_quality_score(
    results[n], max_tokens=128, original_len=len(raw_text)))
print(f"Strategi terpilih: {best_name}")
best_chunks = results[best_name]
chunk_texts = [c.text for c in best_chunks]
chunk_vectors = embedder.encode(chunk_texts, convert_to_numpy=True)

print(f"{len(chunk_texts)} chunk -> embedding shape {chunk_vectors.shape}")
# chunk_texts + chunk_vectors menjadi input untuk nb03 (retrieve + rerank)

## Ringkasan dan Jembatan ke NB-03

Di notebook ini kita telah:

1. **Memahami mengapa ingest PDF itu sulit** — lima jebakan klasik (kolom ganda, tabel, scan, furniture, ligatur).
2. **Membuat pipeline Docling eksplisit** — OCR + TableFormer yang bisa dikonfigurasi, bukan satu kotak hitam.
3. **Membandingkan Docling vs pdfplumber** — dan mengintrospeksi `DoclingDocument` (texts, tables, pictures, iterate_items).
4. **Mendemonstrasikan kasus sulit** — tabel & dokumen scan sebagai bukti konkret nilai Docling.
5. **Mengukur token** dengan tokenizer Qwen2.5-3B-Instruct yang sama dengan model *generator*.
6. **Membandingkan empat pendekatan chunking** — `sentence`, `fixed`, `semantic`, dan HybridChunker sadar-struktur.
7. **Meng-*embed*** *chunk* terpilih menjadi vektor siap-retrieve.

**Kunci takeaway:** Kualitas ingest adalah fondasi RAG. Docling/HybridChunker adalah alternatif sadar-struktur yang melampaui chunking string sederhana untuk dokumen kompleks.

### Selanjutnya: NB-03 — Retrieve & Rerank

nb03 akan memakai `chunk_texts` dan `chunk_vectors` yang kita siapkan di sini sebagai bahan untuk retrieve + rerank. Di sana kita akan belajar:

- **Over-fetch** — mengambil lebih banyak kandidat dari yang dibutuhkan (mis. top-20).
- **Cross-encoder reranking** — menggunakan model *cross-encoder* untuk me-*rerank* kandidat berdasarkan relevansi semantik yang lebih dalam.
- **Mengapa dua tahap ini** jauh lebih baik daripada langsung mengambil top-k dari *vector store*.